# check_valid — validate adapter_model.safetensors

This notebook scans **all** `adapter_model.safetensors` files under `FED-CONS-FINAL/` and reports whether each file is:
- a valid safetensors weights file (loads successfully), or
- likely **ZIP / HTML / Git LFS pointer / corrupted-truncated**.

Run top-to-bottom, then inspect the final summary table (failures first).

In [1]:
# # If needed, install dependencies in the current kernel.
# # (If you're using a fixed conda env, you can skip this cell.)
# !pip install -q safetensors torch pandas

In [7]:
import os
from pathlib import Path

try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

BASE_DIR = Path(NOTEBOOK_DIR)
ROOT = BASE_DIR  # this notebook lives inside FED-CONS-FINAL/

# Optional filters
MAX_FILES = None   # e.g., 10
GLOB = "**/adapter_model.safetensors"

print("ROOT:", ROOT)
print("Exists:", ROOT.exists())

ROOT: d:\Desktop\MIT\CODES\FYP-26\FED-CONS-FINAL - TT
Exists: True


In [8]:
paths = sorted(ROOT.glob(GLOB))
if MAX_FILES:
    paths = paths[: int(MAX_FILES)]

print(f"Found {len(paths)} adapter_model.safetensors files")
for p in paths:
    print("-", p.relative_to(ROOT))

Found 12 adapter_model.safetensors files
- aggregated_adapters\fedavg_error\adapter_model.safetensors
- aggregated_adapters\fedavg_error_v2\adapter_model.safetensors
- aggregated_adapters\fedavg_size\adapter_model.safetensors
- aggregated_adapters\fedavg_size_v1\adapter_model.safetensors
- aggregated_adapters\fedavg_size_v2\adapter_model.safetensors
- aggregated_adapters\fedavg_size_v2_no_ds1000\adapter_model.safetensors
- DS1000\lora_adapters_ds1000\adapter_model.safetensors
- DS1000\lora_adapters_ds1000_v1\adapter_model.safetensors
- HUMANEVAL\lora_adapter_humaneval_try_fix\adapter_model.safetensors
- HUMANEVAL\lora_adapters_humaneval\adapter_model.safetensors
- HUMANEVAL\lora_adapters_x_cross\adapter_model.safetensors
- MBPP\lora_adapters_mbpp\adapter_model.safetensors


In [9]:
import re
from dataclasses import dataclass

ZIP_MAGIC = b"PK\x03\x04"

@dataclass
class FileCheck:
    path: Path
    size_mb: float
    head16_hex: str
    signature: str
    classification: str


def _safe_read_head(path: Path, n: int = 256) -> bytes:
    with path.open("rb") as f:
        return f.read(n)


def _try_read_text(path: Path, n_chars: int = 2000) -> str | None:
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            return f.read(n_chars)
    except Exception:
        return None


def classify_file(path: Path) -> FileCheck:
    size_bytes = path.stat().st_size
    size_mb = size_bytes / (1024**2)
    head = _safe_read_head(path, 256)
    head16_hex = head[:16].hex(" ")

    # Signature heuristics
    if head.startswith(ZIP_MAGIC):
        signature = "zip"
        classification = "zip_archive_renamed_or_downloaded"
        return FileCheck(path, size_mb, head16_hex, signature, classification)

    txt = _try_read_text(path)
    if txt is not None:
        txt_strip = txt.lstrip()
        if "git-lfs.github.com/spec/v1" in txt:
            signature = "git_lfs_pointer"
            classification = "git_lfs_pointer_not_weights"
            return FileCheck(path, size_mb, head16_hex, signature, classification)
        if txt_strip.startswith("<") and ("<!doctype html" in txt_strip.lower() or "<html" in txt_strip.lower() or "href=" in txt_strip.lower()):
            signature = "html"
            classification = "html_download_page_saved_as_safetensors"
            return FileCheck(path, size_mb, head16_hex, signature, classification)

    signature = "binary_unknown"
    classification = "unknown_binary"
    return FileCheck(path, size_mb, head16_hex, signature, classification)


checks = [classify_file(p) for p in paths]
print("Sample classifications:")
for c in checks[: min(10, len(checks))]:
    print(c.path.name, "->", c.signature, "/", c.classification, f"({c.size_mb:.2f} MB)")

Sample classifications:
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)
adapter_model.safetensors -> binary_unknown / unknown_binary (114.25 MB)


In [10]:
from safetensors.torch import load_file

results = []

for c in checks:
    row = {
        "path": str(c.path),
        "rel_path": str(c.path.relative_to(ROOT)),
        "size_mb": round(c.size_mb, 3),
        "head16_hex": c.head16_hex,
        "signature": c.signature,
        "classification": c.classification,
        "load_ok": False,
        "num_tensors": None,
        "error_type": None,
        "error_msg": None,
    }

    try:
        sd = load_file(str(c.path))
        row["load_ok"] = True
        row["num_tensors"] = len(sd)
    except Exception as e:
        row["error_type"] = type(e).__name__
        row["error_msg"] = str(e)

        # If it wasn't clearly ZIP/HTML/LFS, but safetensors failed, it's usually corruption/truncation.
        if row["classification"] == "unknown_binary":
            row["classification"] = "likely_corrupted_or_truncated"

    results.append(row)

print("Done. Loaded OK:", sum(1 for r in results if r["load_ok"]), "/", len(results))

Done. Loaded OK: 12 / 12


In [11]:
def hint_for(row: dict) -> str:
    cls = row.get("classification")
    if cls == "git_lfs_pointer_not_weights":
        return "You have a Git LFS pointer, not weights. Install Git LFS and run: git lfs pull"
    if cls == "zip_archive_renamed_or_downloaded":
        return "This is a ZIP. Unzip it and use the real adapter_model.safetensors inside."
    if cls == "html_download_page_saved_as_safetensors":
        return "This is HTML (bad download). Re-download the file from the source."
    if cls == "likely_corrupted_or_truncated":
        return "Likely corrupted/truncated copy or download. Re-download/re-copy the file."
    if row.get("load_ok"):
        return "OK"
    return "Check path/source; re-download if unsure."


# Prefer a nice table, but don't hard-fail if pandas isn't available.
try:
    import pandas as pd

    df = pd.DataFrame(results)
    df["hint"] = df.apply(lambda r: hint_for(r.to_dict()), axis=1)

    # Failures first
    df = df.sort_values(by=["load_ok", "classification", "size_mb"], ascending=[True, True, False])

    cols = [
        "load_ok",
        "classification",
        "signature",
        "size_mb",
        "num_tensors",
        "error_type",
        "error_msg",
        "rel_path",
        "hint",
    ]
    display(df[cols])
except Exception as e:
    print("pandas table unavailable; falling back to plain text.")
    print("Reason:", e)

    for r in sorted(results, key=lambda x: (x["load_ok"], x["classification"], -x["size_mb"])):
        print("\n-", r["rel_path"])
        print("  load_ok:", r["load_ok"], "| classification:", r["classification"], "| signature:", r["signature"], f"| size_mb: {r['size_mb']}")
        if not r["load_ok"]:
            print("  error:", r["error_type"], "-", r["error_msg"])
        print("  hint:", hint_for(r))

,load_ok,classification,signature,size_mb,num_tensors,error_type,error_msg,rel_path,hint
0,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_error\adapter_model...,OK
1,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_error_v2\adapter_mo...,OK
2,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_size\adapter_model....,OK
3,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_size_v1\adapter_mod...,OK
4,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_size_v2\adapter_mod...,OK
5,True,unknown_binary,binary_unknown,114.252,504,None,None,aggregated_adapters\fedavg_size_v2_no_ds1000\a...,OK
6,True,unknown_binary,binary_unknown,114.252,504,None,None,DS1000\lora_adapters_ds1000\adapter_model.safe...,OK
7,True,unknown_binary,binary_unknown,114.252,504,None,None,DS1000\lora_adapters_ds1000_v1\adapter_model.s...,OK
8,True,unknown_binary,binary_unknown,114.252,504,None,None,HUMANEVAL\lora_adapter_humaneval_try_fix\adapt...,OK
9,True,unknown_binary,binary_unknown,114.252,504,None,None,HUMANEVAL\lora_adapters_humaneval\adapter_mode...,OK
